# Prepare the Settings

In [ ]:
import os, torch
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("cuda.is_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())


In [ ]:
import os 
import numpy as np

import pandas as pd
import json
import torch
from model import prepare_model
from model import Model_Wrapper
from omegaconf import OmegaConf
import tempfile

from transformers import AutoModel, AutoProcessor,BitsAndBytesConfig

from sklearn.preprocessing import OrdinalEncoder, Normalizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold, GroupShuffleSplit, GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix, matthews_corrcoef, average_precision_score, f1_score

from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
import torch.nn.functional as F

from transformers import BertTokenizer, AutoModel, AutoTokenizer, AutoConfig

# This conf file is required to load the pretrained weights. 
config = OmegaConf.create({
    "saved_checkpoints" : './group_attn_run2/checkpoint.pt',
    "image_feature_file" : './data/UKB/Macular_Measurement.csv',
    "train_json" : "captions_train.json",
    "val_json" : "captions_val.json",
    "test_json" : "captions_test2.json",
    "mode": 'train',
    "logs" : 'logs',
    "logs_name": "training_logs.txt",
    "use_distance": False,
    "use_assign": False,
    "vision_model_checkpoint":'./RETFound_MAE/RETFound_cfp_weights.pth', 
    "vision_model_output_dim": 1024,
    "text_model_output_dim": 1024,
    "context_length": 512, 
    "proj_dim": 1024,
    "dropout": 0,
    "num_train_epochs": 3, 
    "batch_size": 16,
    "gradient_accumulation_steps" : 1,
    "loss": 'improved',
    "optimizer": {
        "params":{
            'eps': 1.0e-07, 
            'lr': 1e-5, 
            'weight_decay': 0.1}
    },
    "n_gpu":1,
    "logging_steps" : 10,
    "save_steps" : 10
                          })

In [ ]:
# For deterministic settings
def set_seed(seed=0):
    import os
    import random
    import numpy as np
    import torch

    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Settings

* test_model: model used in the evaluation.
* task: Alzheimer's Disease (AD) or dementia (DE)

In [ ]:
test_model = 'reveal'
ml_model = 'svm'
task = "AD" # task should be either AD or DE
nc_ratio = 15 # the matched control should have been set to 15 to sample enough number of nc subjects (approximately 12)

## Read Dataframe and models

In [ ]:
with open("./data/UKB/captions_test2.json", "r") as a:
    json_data = json.load(a)
test_df = pd.DataFrame(json_data['images'])

def _transform(n_px):
    """
    Defines a transformation pipeline for image preprocessing.

    Args:
        n_px (int): The target image size for resizing.

    Returns:
        torchvision.transforms.Compose: A composed transformation pipeline.
    """
    return Compose([
        Resize(n_px, interpolation=Image.BICUBIC),  # Resize image
        lambda image: image.convert("RGB"),  # Convert to RGB
        ToTensor(),  # Convert image to tensor
        Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),  # Normalize using COCO mean & std
    ])

transform = _transform(224)
device = "cuda" if torch.cuda.is_available() else "cpu"

# RETFound
if test_model == 'retfound': 
    vision_model = prepare_model('./RETFound_MAE/RETFound_cfp_weights.pth', 'vit_large_patch16').to(device)
    text_model = AutoModel.from_pretrained('UFNLP/gatortronS').to(device)
    tokenizer = AutoTokenizer.from_pretrained('UFNLP/gatortronS')

# REVEAL
if test_model == 'reveal':
    model = Model_Wrapper(config)
    checkpoint = torch.load('./saved_checkpoints/use_GACL_true/best_model.pt')
    model.load_state_dict(checkpoint['model_state_dict'])
    tokenizer = AutoTokenizer.from_pretrained('UFNLP/gatortronS')
    model.to(device)

# PMC_CLIP
if test_model == 'pmc_clip':
    import open_clip
    model, _, preprocess_val = open_clip.create_model_and_transforms('hf-hub:ryanyip7777/pmc_vit_l_14', cache_dir="./cache/dir")
    tokenizer = open_clip.get_tokenizer('hf-hub:ryanyip7777/pmc_vit_l_14', cache_dir="./cache/dir")
    vision_model = model.visual
    text_model = model.transformer
    
    vision_model.to(device)
    vision_model.eval()
    
    text_model.to(device)
    text_model.eval()
    tokenizer.context_length=768

# BiomedCLIP
if test_model == 'biomedclip':
    from open_clip import create_model_from_pretrained, get_tokenizer
    model, preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224', cache_dir="./cache/dir")
    tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224', cache_dir="./cache/dir")
    
    vision_model = model.visual
    text_model = model.text
    
    vision_model.to(device)
    vision_model.eval()
    
    text_model.to(device)
    text_model.eval()
    
# KeepFIT-CFP    
if test_model == 'keepfit':
    import torch
    from keepfit.modeling.model import KeepFITModel  # adjust import path based on repo

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # instantiate model (vision + text exports)
    model = KeepFITModel(weights_path='./keepfit/KeepFIT (FFA-IR+MM).pth')  # pass args consistent with the model definition: modality=‘CFP’, etc
    model = model.to(device)
    model.eval()

    vision_model = model.vision_model 
    text_model   = model.text_model

    vision_model.to(device).eval()
    text_model.to(device).eval()
    
    tokenizer = text_model.tokenizer
    preprocess_image = model.preprocess_image
    preprocess_text = model.preprocess_text

## Load the Sampled EIDs. I sampled the stratified control set from the test set subject participants.

In [ ]:
import json
with open("/data/UKB/captions_test2.json", "r") as a:
    data = json.load(a)
test_df_text = pd.DataFrame(data['annotations'])

if task == 'AD':
    with open("/data/UKB/test_AD_positive_eid.json") as b:
        ad_id = json.load(b)
    with open("data/UKB/test_AD_control_eid.json") as c:
        nc_id = json.load(c)
elif task == 'DE':
    with open("/data/UKB/test_DE_positive_eid.json") as b:
        ad_id = json.load(b)
    with open("/data/UKB/test_DE_control_eid.json") as c:
        nc_id = json.load(c)

In [ ]:
import numpy as np
import pandas as pd

subject_info = pd.read_csv("/data/UKB/subject_info.csv")

# Convert to sets for fast comparison
ad_set = set(ad_id)
nc_set = set(nc_id)

# Find overlapping elements
overlap = ad_set & nc_set
print(f"Number of overlapping IDs: {len(overlap)}")

if overlap:
    print("Overlapping IDs:", list(overlap)[:10])  # preview first 10

# Remove overlaps from nc_id
nc_id_clean = [i for i in nc_id if i not in ad_set]

print(f"Original nc_id count: {len(nc_id)}")
print(f"Cleaned nc_id count: {len(nc_id_clean)}")

nc_df = subject_info[subject_info['eid'].isin(nc_id_clean)]
ad_df = subject_info[subject_info['eid'].isin(ad_id)]

## Simple statistical set used in the sampling process to confirm Age and Sex is not significantly different between 2 groups.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact, ttest_ind
from statsmodels.stats.multitest import multipletests

# --- helpers ---
def _normalize_sex(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip().str.lower()
    out = s.replace({
        "male": "M", "m": "M", "1": "M",
        "female": "F", "f": "F", "0": "F"
    })
    # treat textual NaNs as real NaN
    return out.where(~s.isin(["nan", "none", "na", ""]), other=np.nan)

def _cramers_v_from_ct(ct: pd.DataFrame) -> float:
    chi2, _, _, _ = chi2_contingency(ct.values, correction=False)
    n = ct.values.sum()
    r, k = ct.shape
    denom = max(1, min(r - 1, k - 1))
    return float(np.sqrt((chi2 / n) / denom)) if n > 0 and denom > 0 else np.nan

def _cohen_d_welch(x, y):
    x = np.asarray(x); y = np.asarray(y)
    x = x[np.isfinite(x)]; y = y[np.isfinite(y)]
    if x.size < 2 or y.size < 2:
        return np.nan, np.nan
    vx, vy = x.var(ddof=1), y.var(ddof=1)
    d = (x.mean() - y.mean()) / np.sqrt((vx + vy) / 2.0)
    df = x.size + y.size - 2
    J = 1.0 - (3.0 / (4.0 * df - 1.0)) if df > 1 else 1.0
    return float(d), float(d * J)

def _mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)  # (B, L, 1)
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counts

# --- integrated function ---
def compare_groups(
    ad_df: pd.DataFrame,
    nc_df: pd.DataFrame,
    categorical_cols: list,
    continuous_cols: list,
    fdr: bool = True
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Robust categorical comparisons
    cat_rows = []
    for col in categorical_cols:
        a = ad_df[col].copy()
        n = nc_df[col].copy()

        # optional normalization for 'sex'
        if col.lower() == "sex":
            a, n = _normalize_sex(a), _normalize_sex(n)

        ad_counts = a.value_counts(dropna=False)
        nc_counts = n.value_counts(dropna=False)
        cats = ad_counts.index.union(nc_counts.index)

        ad_vec = ad_counts.reindex(cats, fill_value=0).astype(int)
        nc_vec = nc_counts.reindex(cats, fill_value=0).astype(int)

        ct = pd.DataFrame([ad_vec, nc_vec], index=["AD", "NC"])
        ct = ct.loc[:, ct.sum(axis=0) > 0]  # drop all-zero columns

        row = {"variable": col, "method": None, "chi2": np.nan, "dof": np.nan,
               "p_value": np.nan, "effect": np.nan, "levels": int(ct.shape[1])}

        if ct.shape[1] == 0:
            row.update({"method": "none (no levels)"})
        elif ct.shape[1] == 1:
            row.update({"method": "constant (2x1)", "p_value": 1.0, "effect": 0.0})
        elif ct.shape == (2, 2):
            chi2, p_chi2, _, expected = chi2_contingency(ct.values, correction=False)
            if (expected < 5).any():
                _, p_fisher = fisher_exact(ct.values)
                row.update({"method": "fisher_exact", "p_value": float(p_fisher)})
                # compute phi from chi2 (uses chi2 from the same table)
                n_tot = ct.values.sum()
                phi = np.sqrt(chi2 / n_tot) if n_tot > 0 else np.nan
                row["effect"] = float(phi)
            else:
                row.update({"method": "chi2", "chi2": float(chi2), "dof": 1, "p_value": float(p_chi2)})
                n_tot = ct.values.sum()
                phi = np.sqrt(chi2 / n_tot) if n_tot > 0 else np.nan
                row["effect"] = float(phi)
        else:
            chi2, p, dof, _ = chi2_contingency(ct.values, correction=False)
            row.update({"method": "chi2", "chi2": float(chi2), "dof": int(dof),
                        "p_value": float(p), "effect": float(_cramers_v_from_ct(ct))})

        cat_rows.append(row)

    cat_df = pd.DataFrame(cat_rows)
    if not cat_df.empty and fdr:
        cat_df["p_fdr"] = multipletests(cat_df["p_value"].fillna(1.0), method="fdr_bh")[1]
    if not cat_df.empty:
        cat_df = cat_df.sort_values("p_value", na_position="last")

    # Continuous comparisons (Welch's t-test)
    cont_rows = []
    for col in continuous_cols:
        a = pd.to_numeric(ad_df[col], errors="coerce").dropna().values
        n = pd.to_numeric(nc_df[col], errors="coerce").dropna().values
        if a.size >= 2 and n.size >= 2:
            t, p = ttest_ind(a, n, equal_var=False, nan_policy="omit")
            d, g = _cohen_d_welch(a, n)  # AD - NC effect size
        else:
            t, p, d, g = np.nan, np.nan, np.nan, np.nan

        cont_rows.append({
            "variable": col, "t_stat": t, "p_value": p,
            "cohen_d": d, "hedges_g": g,
            "mean_AD": float(np.nanmean(a)) if a.size else np.nan,
            "mean_NC": float(np.nanmean(n)) if n.size else np.nan,
            "n_AD": int(a.size), "n_NC": int(n.size)
        })

    cont_df = pd.DataFrame(cont_rows)
    if not cont_df.empty and fdr:
        cont_df["p_fdr"] = multipletests(cont_df["p_value"].fillna(1.0), method="fdr_bh")[1]
    if not cont_df.empty:
        cont_df = cont_df.sort_values("p_value", na_position="last")

    return cat_df, cont_df

# --- Get the statistics of the split ---
categorical_cols = ["sex"]
continuous_cols  = ["age"]
cat_stats, cont_stats = compare_groups(ad_df, nc_df, categorical_cols, continuous_cols, fdr=True)
print(cat_stats)
print(cont_stats)


## Embedding Extraction

In [ ]:
# ---------------------------------------------------------
# Feature extraction session. Based on the sampled ids, 
# the image and text embedding is extracted. 
# ---------------------------------------------------------

# Seed define
seeds = [0,1,2,3,4,5,6,7,8,9]

# To store the results
roc_auc_list = []
bal_acc_list = []
sen_list = []
spec_list = []
prec_list = []
f1_list = []
mcc_list = []
auprc_list = []

# To store AD (label=1) eids per seed
ad_correct_ids_all_seeds = []      # AD correctly classified (TP)
ad_incorrect_ids_all_seeds = []    # AD incorrectly classified (FN)

# Define the transform
to_tensor = ToTensor()

# Get NC and AD image data.
nc = test_df[test_df['id'].isin(nc_id)]
ad = test_df[test_df['id'].isin(ad_id)]

# Label assigning    
ad['label'] = 1
nc['label'] = 0
data_image = pd.concat([ad, nc], ignore_index=True)

# Get NC and AD text data.
nc = test_df_text[test_df_text['image_id'].isin(nc_id)]
ad = test_df_text[test_df_text['image_id'].isin(ad_id)]
    
data_text = pd.concat([ad, nc], ignore_index=True)

# Separate the data into input and output.
label = data_image.pop('label')  # y

processed_data = pd.DataFrame()

# ---------------------------------------------------------
# Embedding generation loop
# ---------------------------------------------------------
for i in range(len(data_image)):
    img_path = data_image.iloc[i]['path']
    
    img_path = '/UKB/data/Eye/'
    image = Image.open(img_path)

    if test_model == 'retfound':       
        image = transform(image)
        text = data_text.iloc[i]['caption']
        text = tokenizer(
            text,
            return_tensors="pt",
            padding='max_length',
            max_length=config.context_length,
            truncation=True
        ).to(device)

        image_embeds = vision_model(image.unsqueeze(0).to(device))
        text_embeds = _mean_pool(text_model(**text).last_hidden_state, text['attention_mask'])
            
        image_embeds = image_embeds / image_embeds.norm(dim=1, keepdim=True)
        text_embeds = text_embeds / text_embeds.norm(dim=1, keepdim=True)

        feat = torch.cat([image_embeds, text_embeds], dim=1)
        output = feat.detach().cpu()
            
    elif test_model == 'reveal':
        image = transform(image)
        text = data_text.iloc[i]['caption']
        text = tokenizer(
            text,
            return_tensors="pt",
            padding='max_length',
            max_length=config.context_length,
            truncation=True
        ).to(device)

        image_embeds, text_embeds = model(image.unsqueeze(0).to(device), text)
            
        image_embeds = image_embeds / image_embeds.norm(dim=1, keepdim=True)
        text_embeds = text_embeds / text_embeds.norm(dim=1, keepdim=True)

        feat = torch.cat([image_embeds, text_embeds], dim=1)
        output = feat.detach().cpu()

    elif test_model == 'pmc_clip':
        image = preprocess_val(image)
        image_embeds = vision_model.forward(image.unsqueeze(0).to(device)).detach().cpu()
            
        text = data_text.iloc[i]['caption']
        text = tokenizer([text]).to(device)
        text_embeds = text_model(text.float()).detach().cpu()
            
        image_embeds = image_embeds / image_embeds.norm(dim=1, keepdim=True)
        text_embeds = text_embeds / text_embeds.norm(dim=1, keepdim=True)
            
        feat = torch.cat([image_embeds, text_embeds], dim=1)
        output = feat.detach().cpu()
            
    elif test_model == 'biomedclip':
        image = preprocess(image).to(device)
        image_embeds = vision_model.forward(image.unsqueeze(0)).detach().cpu()
            
        text = data_text.iloc[i]['caption']
        text = tokenizer(text).to(device)
        text_embeds = text_model(text).detach().cpu()
            
        image_embeds = image_embeds / image_embeds.norm(dim=1, keepdim=True)
        text_embeds = text_embeds / text_embeds.norm(dim=1, keepdim=True)
 
        feat = torch.cat([image_embeds, text_embeds], dim=1)
        output = feat.detach().cpu()
        
    elif test_model == 'keepfit':
        image = np.array(image)
        image = preprocess_image(image).to(device)
            
        image_embeds = vision_model.forward(image).detach().cpu()
            
        text = data_text.iloc[i]['caption']
        text_input_ids, text_attention_mask = preprocess_text(text)
        text_embeds = text_model(text_input_ids, text_attention_mask).detach().cpu()
        text_embeds = text_embeds[0, :].unsqueeze(0)
            
        image_embeds = image_embeds / image_embeds.norm(dim=1, keepdim=True)
        text_embeds = text_embeds / text_embeds.norm(dim=1, keepdim=True)
 
        feat = torch.cat([image_embeds, text_embeds], dim=1)
        output = feat.detach().cpu()

    processed_data = pd.concat([processed_data, pd.DataFrame(output)], ignore_index=True)

## Classifier Training Process

In [ ]:
# ---------------------------------
# Seeded evaluation loop
# ---------------------------------
for seed in seeds:
    set_seed(seed)

    # --------------------------------------------------------------------
    # EID-LEVEL TRAIN/TEST SPLIT
    # --------------------------------------------------------------------
    processed_data = processed_data.reset_index(drop=True)
    label = label.reset_index(drop=True)
    eids = data_image['id'].reset_index(drop=True)  # or 'eid' if that’s your column name

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, test_idx = next(gss.split(processed_data, label, groups=eids))

    X_train = processed_data.iloc[train_idx]
    X_test = processed_data.iloc[test_idx]
    y_train = label.iloc[train_idx]
    y_test = label.iloc[test_idx]
    groups_train = eids.iloc[train_idx]

    #print("y_train distribution:")
    #print(y_train.value_counts())

    #print("\ny_test distribution:")
    #print(y_test.value_counts())

    # --------------------------------------------------------------------
    # EID-LEVEL CROSS-VALIDATION (GroupKFold)
    # --------------------------------------------------------------------
    cv = GroupKFold(n_splits=5)

    # class weights
    counts = y_train.value_counts()
    nc_ratio = counts[0] / counts[1]   # ratio

    weights = {0: 1.0, 1: nc_ratio}  # up-weight class "1" if needed
    print(weights)

    # --------------------------------------------------------------------
    # Model & hyperparameter grid
    # --------------------------------------------------------------------
    if ml_model == 'svm':
        param_grid = {
            'C': [100, 1000, 10000],
            'kernel': ['linear', 'rbf'],
            'gamma': ['auto','scale', 0.1, 1],
            'probability': [True],
        }
        estimator = SVC(probability=True, class_weight=weights)

    elif ml_model == 'lr':
        param_grid = {
            'C': [0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear'],
        }
        estimator = LogisticRegression(max_iter=1000, class_weight=weights)

    # GridSearchCV with group-based CV
    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv.split(X_train, y_train, groups=groups_train),
        scoring='roc_auc',
        n_jobs=-1,
    )

    grid_search.fit(X_train, y_train)

    print("Best Parameters:", grid_search.best_params_)
    print("Best Score (CV AUC):", grid_search.best_score_)

    # --------------------------------------------------------------------
    # Evaluation on held-out eid-level test set
    # --------------------------------------------------------------------
    y_pred = grid_search.best_estimator_.predict(X_test)
    y_proba = grid_search.best_estimator_.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # recall / TPR
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0  # TNR

    mcc = matthews_corrcoef(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auprc = average_precision_score(y_test, y_proba)

    result_acc = balanced_accuracy_score(y_test, y_pred)
    result_auc = roc_auc_score(y_test, y_proba)

    bal_acc_list.append(result_acc)
    roc_auc_list.append(result_auc)
    sen_list.append(sensitivity)
    spec_list.append(specificity)
    prec_list.append(precision)
    mcc_list.append(mcc)
    f1_list.append(f1)
    auprc_list.append(auprc)

    # ---------------------------------------------------------
    #   AD (label=1) eids: correctly vs incorrectly classified
    # ---------------------------------------------------------
    # Map test indices back to original data_image to get eids
    test_eids = data_image.loc[X_test.index, "id"]   # eid for each sample in X_test

    ad_mask = (y_test == 1)

    # AD correctly classified: true label 1 and predicted 1  (TP)
    ad_correct_ids = test_eids[(ad_mask) & (y_pred == 1)].tolist()

    # AD incorrectly classified: true label 1 but predicted 0 (FN)
    ad_incorrect_ids = test_eids[(ad_mask) & (y_pred == 0)].tolist()

    ad_correct_ids_all_seeds.append({
        "seed": seed,
        "ad_correct_ids": ad_correct_ids
    })

    ad_incorrect_ids_all_seeds.append({
        "seed": seed,
        "ad_incorrect_ids": ad_incorrect_ids
    })

In [ ]:
print('The average AUROC on the test set is {}'.format(np.mean(roc_auc_list)))
print('The average Accuracy on the test set is {}'.format(np.mean(bal_acc_list)))
print('The average Sensitivity on the test set is {}'.format(np.mean(sen_list)))
print('The average Specificity on the test set is {}'.format(np.mean(spec_list)))
print('The average Precision on the test set is {}'.format(np.mean(prec_list)))
print('The average F1 on the test set is {}'.format(np.mean(f1_list)))
print('The average MCC on the test set is {}'.format(np.mean(mcc_list)))
print('The average AUPRC on the test set is {}'.format(np.mean(auprc_list)))

In [ ]:
print('bal_acc')
print(str(np.mean(bal_acc_list)) + str( np.std(bal_acc_list)))
print('roc_auc')
print(str(np.mean(roc_auc_list)) + str(np.std(roc_auc_list)))
print('f1')
print(str(np.mean(f1_list)) + str(np.std(f1_list)))
print('mcc')
print(str(np.mean(mcc_list)) + str(np.std(mcc_list)))

In [ ]:
import pandas as pd

df_ad_correct = pd.DataFrame([
    {"seed": e["seed"], "eid": eid}
    for e in ad_correct_ids_all_seeds
    for eid in e["ad_correct_ids"]
])
df_ad_correct.to_csv("ad_correct_ids_per_seed.csv", index=False)

df_ad_incorrect = pd.DataFrame([
    {"seed": e["seed"], "eid": eid}
    for e in ad_incorrect_ids_all_seeds
    for eid in e["ad_incorrect_ids"]
])
df_ad_incorrect.to_csv("ad_incorrect_ids_per_seed.csv", index=False)
